In [3]:
words = open('names.txt', 'r').read().splitlines()

In [4]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [5]:
len(words)

32033

In [6]:
min([len(w) for w in words])

2

In [7]:
max(len(w) for w in words)

15

In [8]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1
        # print(ch1, ch2)

In [9]:
sorted(b.items(), key = lambda kv: -kv[1])

[(('n', '<E>'), 6763),
 (('a', '<E>'), 6640),
 (('a', 'n'), 5438),
 (('<S>', 'a'), 4410),
 (('e', '<E>'), 3983),
 (('a', 'r'), 3264),
 (('e', 'l'), 3248),
 (('r', 'i'), 3033),
 (('n', 'a'), 2977),
 (('<S>', 'k'), 2963),
 (('l', 'e'), 2921),
 (('e', 'n'), 2675),
 (('l', 'a'), 2623),
 (('m', 'a'), 2590),
 (('<S>', 'm'), 2538),
 (('a', 'l'), 2528),
 (('i', '<E>'), 2489),
 (('l', 'i'), 2480),
 (('i', 'a'), 2445),
 (('<S>', 'j'), 2422),
 (('o', 'n'), 2411),
 (('h', '<E>'), 2409),
 (('r', 'a'), 2356),
 (('a', 'h'), 2332),
 (('h', 'a'), 2244),
 (('y', 'a'), 2143),
 (('i', 'n'), 2126),
 (('<S>', 's'), 2055),
 (('a', 'y'), 2050),
 (('y', '<E>'), 2007),
 (('e', 'r'), 1958),
 (('n', 'n'), 1906),
 (('y', 'n'), 1826),
 (('k', 'a'), 1731),
 (('n', 'i'), 1725),
 (('r', 'e'), 1697),
 (('<S>', 'd'), 1690),
 (('i', 'e'), 1653),
 (('a', 'i'), 1650),
 (('<S>', 'r'), 1639),
 (('a', 'm'), 1634),
 (('l', 'y'), 1588),
 (('<S>', 'l'), 1572),
 (('<S>', 'c'), 1542),
 (('<S>', 'e'), 1531),
 (('j', 'a'), 1473),
 (

In [10]:
import torch

In [11]:
N = torch.zeros((27, 27), dtype = torch.int32)

In [12]:
chars = list(sorted(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i+1:s for i, s in enumerate(chars)}
itos[0] = '.'

In [13]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1, ix2 = stoi[ch1], stoi[ch2]
        N[ix1, ix2] += 1

In [14]:
N[0]

tensor([   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
        1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
         134,  535,  929], dtype=torch.int32)

In [15]:
p = N[0].float()
p = p / p.sum()
p

tensor([0.0000, 0.1377, 0.0408, 0.0481, 0.0528, 0.0478, 0.0130, 0.0209, 0.0273,
        0.0184, 0.0756, 0.0925, 0.0491, 0.0792, 0.0358, 0.0123, 0.0161, 0.0029,
        0.0512, 0.0642, 0.0408, 0.0024, 0.0117, 0.0096, 0.0042, 0.0167, 0.0290])

In [16]:
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
ix

3

In [17]:
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
p = p / p.sum()
p

tensor([0.6064, 0.3033, 0.0903])

In [41]:
N.float().sum(dim=1, keepdim=True).shape

torch.Size([27, 1])

In [55]:
# model smoothing: add 1 so we don't have log(0) = inf
# add more -> more smooth graph
P = (N+1).float()
P /= P.sum(dim=1, keepdim=True)

In [56]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(10):
    ix = 0
    out = []
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break

    print(''.join(out))


cexze.
momasurailezitynn.
konimittain.
llayn.
ka.
da.
staiyaubrtthrigotai.
moliellavo.
ke.
teda.


In [58]:
# log(a*b*c) = log(a) + log(b) + log(c)

log_likelihood = 0
n = 0
# for w in words[:3]:
for w in ["andrejq"]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1, ix2 = stoi[ch1], stoi[ch2]
        N[ix1, ix2] += 1
        prob = P[ix1, ix2]
        logprob = torch.log(prob)
        log_likelihood += logprob
        n += 1
        print(f"{ch1}{ch2}: {prob:.4f} {logprob:.4f}")

print(f"{log_likelihood=}")
nll = -log_likelihood
print(f"{nll=}")
print(f"{nll/n}")

.a: 0.1377 -1.9827
an: 0.1603 -1.8309
nd: 0.0384 -3.2594
dr: 0.0770 -2.5646
re: 0.1334 -2.0143
ej: 0.0027 -5.9007
jq: 0.0003 -7.9817
q.: 0.0970 -2.3331
log_likelihood=tensor(-27.8674)
nll=tensor(27.8674)
3.4834272861480713


## Neural Network for Bigram

In [63]:
# Create training dset for bigram (x, y)
xs, ys = [], []
for w in words[:1]:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1, ix2 = stoi[ch1], stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)

In [91]:
import torch.nn.functional as F

xenc = F.one_hot(xs, num_classes=27).float()
xenc

tensor([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [125]:
 # input to the neural net: one-hot encoding
xenc = F.one_hot(xs, num_classes=27).float()

# initialize 27 neurons, each with 27 weights
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

# forward pass
logits = xenc @ W # log-counts
# softmax
counts = logits.exp() # counts, equivalant to N
probs = counts / counts.sum(dim = 1, keepdims=True) # prob dist. for next char
loss = -probs[torch.arange(5), ys].log().mean()


In [126]:
# backward pass
W.grad = None # set gradient to 0
loss.backward()
W.data += -0.01 * W.grad

In [127]:
# --- Optimization ---

In [140]:
# --- 1. Dataset Setup ---

# Mapping characters to integers (26 letters + '.' for start/end)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

# Create the dataset
xs, ys = [], []
for w in words:
    # add special start/end token
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of examples: ', num)

number of examples:  228146


In [154]:
# --- 2. Initialize the 'Network' ---
g = torch.Generator().manual_seed(2147483647)
# 27 characters in, 27 characters out
W = torch.randn((27, 27), generator=g, requires_grad=True)

In [ ]:
# --- 3. Gradient Descent ---
for k in range(30):
    
    # Forward Pass
    # Input to the network: one-hot encoding
    xenc = F.one_hot(xs, num_classes=27).float() 
    logits = xenc @ W # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    probs = counts / counts.sum(1, keepdims=True) # probabilities for next character
    
    # Negative Log Likelihood (NLL) Loss
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W**2).mean()
    print(f"Iteration {k}: loss = {loss.item()}")

    # Backward Pass
    W.grad = None # set to zero the gradient
    loss.backward()

    # Update
    W.data += -10 * W.grad

Iteration 0: loss = 2.498103380203247
Iteration 1: loss = 2.488374710083008
Iteration 2: loss = 2.481330394744873
Iteration 3: loss = 2.476222515106201
Iteration 4: loss = 2.472583293914795
Iteration 5: loss = 2.470055341720581
Iteration 6: loss = 2.4683480262756348
Iteration 7: loss = 2.4672257900238037
Iteration 8: loss = 2.466503143310547
Iteration 9: loss = 2.4660451412200928
Iteration 10: loss = 2.46575665473938
Iteration 11: loss = 2.465574026107788
Iteration 12: loss = 2.465456485748291
Iteration 13: loss = 2.465378522872925
Iteration 14: loss = 2.4653244018554688
Iteration 15: loss = 2.4652841091156006
Iteration 16: loss = 2.46525239944458
Iteration 17: loss = 2.4652259349823
Iteration 18: loss = 2.465203046798706
Iteration 19: loss = 2.465181827545166
Iteration 20: loss = 2.4651620388031006
Iteration 21: loss = 2.4651429653167725
Iteration 22: loss = 2.4651246070861816
Iteration 23: loss = 2.465106725692749
Iteration 24: loss = 2.4650893211364746
Iteration 25: loss = 2.4650714

In [158]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(2147483647)

for i in range(5):
    
    out = []
    ix = 0 # start with the '.' token
    while True:
        
        # ----------
        # BEFORE:
        # p = P[ix]
        # ----------
        # NOW:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W # predict log-counts
        counts = logits.exp() # counts, equivalent to N
        p = counts / counts.sum(1, keepdims=True) # probabilities for next character
        # ----------
        
        # Sample from the distribution p
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        
        # If we sample the '.' token (0), the word is finished
        if ix == 0:
            break
            
    print(''.join(out))

cexze.
momakurailezityha.
konimittain.
llayn.
ka.
